# Python Syntax, Indentation, Comments, and Code Structure

**Topic:** `1.3` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Strip a trailing comment from a line

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** strings, comments, parsing

**Explanation and reasoning**

The scan tracks the active quote character so a '#' inside a string is never treated as a comment. Tracking the previous character distinguishes an escaped quote from a closing one, and returning the prefix keeps the code that precedes the comment intact.

In [ ]:
def strip_comments(line: str) -> str:
    quote = None
    previous = ''
    for index, ch in enumerate(line):
        if quote:
            if ch == quote and previous != '\\':
                quote = None
        elif ch in ('"', "'"):
            quote = ch
        elif ch == '#':
            return line[:index].rstrip()
        previous = ch
    return line.rstrip()

**Complexity**

Time O(n) in the line length; space O(1) beyond the result.

**Edge cases and failure modes**

A comment-only line returns an empty string; a trailing backslash escape is respected.

**Alternative approaches**

tokenize handles this correctly but requires a full source context.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert strip_comments('x = 1  # note') == 'x = 1'
assert strip_comments("s = '# not a comment'") == "s = '# not a comment'"
assert strip_comments('# only') == ''

---

## Solution 2 — Measure indentation width

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** strings, indentation, counting

**Explanation and reasoning**

Comparing lengths recovers the indentation without mutating the line. The tab check comes first so a mixed line reports -1 rather than a misleading count, and the blank case is handled before any counting so whitespace-only lines never produce a width.

In [ ]:
def indent_width(line: str) -> int:
    stripped = line.lstrip()
    if not stripped:
        return 0
    lead = line[:len(line) - len(stripped)]
    if '\t' in lead:
        return -1
    return len(lead)

**Complexity**

Time O(n) in the line length; space O(n) for the stripped copy.

**Edge cases and failure modes**

A whitespace-only line returns 0; a line with trailing tabs is unaffected.

**Alternative approaches**

re.match(r'^[ ]*', line) avoids creating the stripped copy.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert indent_width('    return 1') == 4
assert indent_width('\treturn 1') == -1
assert indent_width('   ') == 0

---

## Solution 3 — Normalise a block to a fixed indent width

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** indentation, strings, mapping

**Explanation and reasoning**

The first pass assigns each distinct original indent a depth, in the order the widths are first seen, so nesting is preserved without counting block headers. The second pass rewrites each line using that depth, which means input that is already correct comes back unchanged.

In [ ]:
def reindent(lines, width: int = 4) -> list[str]:
    depth_of = {}
    for line in lines:
        if not line.strip():
            continue
        lead = len(line) - len(line.lstrip(' '))
        depth_of.setdefault(lead, len(depth_of))
    out = []
    for line in lines:
        if not line.strip():
            out.append('')
            continue
        lead = len(line) - len(line.lstrip(' '))
        out.append(' ' * (depth_of[lead] * width) + line.lstrip(' '))
    return out

**Complexity**

Time O(n) in the number of lines; space O(k) for k distinct widths.

**Edge cases and failure modes**

Blank lines become empty strings; already-correct input is unchanged.

**Alternative approaches**

Walking the AST gives exact nesting but requires valid input.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert reindent(['if ready:', '  return 1']) == ['if ready:', '    return 1']
assert reindent(['a', '    b']) == ['a', '    b']

---

## Solution 4 — Build a multi-line banner

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** strings, f-strings, implicit joining

**Explanation and reasoning**

The whole banner is one expression wrapped in parentheses, so the three parts sit on separate source lines without any concatenation operator. The '>12' format spec right-aligns each field, which is what produces the fixed-width column layout.

In [ ]:
def banner(robot: str, level: str) -> str:
    return (
        f'{robot:>12}\n'
        f'{level:>12}\n'
        f"{'READY':>12}"
    )

**Complexity**

Time O(n) in the input lengths; space O(n).

**Edge cases and failure modes**

Names longer than twelve characters widen the column rather than truncating.

**Alternative approaches**

A triple-quoted f-string is shorter but keeps the newlines in the literal.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert banner('rover-01', 'M1').splitlines() == ['    rover-01', '          M1', '       READY']
assert len(banner('r', 'M1').splitlines()) == 3

---

## Solution 5 — Count code, comment and blank lines

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** tokenize, strings, classification

**Explanation and reasoning**

Branching on the stripped form makes the three categories mutually exclusive, so every line increments exactly one counter. Computing total from the line list rather than by summing keeps the invariant total equals the sum of the parts even when the input is empty.

In [ ]:
def line_census(source: str) -> dict:
    counts = {'code': 0, 'comment': 0, 'blank': 0}
    lines = source.splitlines()
    for line in lines:
        stripped = line.strip()
        if not stripped:
            counts['blank'] += 1
        elif stripped.startswith('#'):
            counts['comment'] += 1
        else:
            counts['code'] += 1
    counts['total'] = len(lines)
    return counts

**Complexity**

Time O(n) in the number of lines; space O(1).

**Edge cases and failure modes**

A trailing newline does not create an extra line because splitlines handles it.

**Alternative approaches**

tokenize distinguishes comments precisely but treats docstrings as strings, not comments.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert line_census('# note\nx = 1\n\n') == {'code': 1, 'comment': 1, 'blank': 1, 'total': 3}
assert line_census('')['total'] == 0

---

## Solution 6 — Report indentation problems

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** indentation, validation, strings

**Explanation and reasoning**

The continue after the tab check enforces at most one problem per line, which keeps the report unambiguous. Blank lines are skipped before any measurement so trailing whitespace at the end of a file never produces noise.

In [ ]:
def check_indentation(source: str, width: int = 4) -> list[str]:
    problems: list[str] = []
    for number, line in enumerate(source.splitlines(), start=1):
        if not line.strip():
            continue
        lead = line[:len(line) - len(line.lstrip())]
        if '\t' in lead:
            problems.append(f'line {number}: tab in indentation')
            continue
        spaces = len(lead)
        if spaces % width:
            problems.append(
                f'line {number}: indent {spaces} is not a multiple of {width}'
            )
    return problems

**Complexity**

Time O(n) in the number of lines; space O(p) for p problems.

**Edge cases and failure modes**

Clean source returns an empty list; a tab line is reported only once.

**Alternative approaches**

Delegating to indent_width from exercise 2 keeps the measurement in one place.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert check_indentation('if ready:\n  return 1\n') == ["line 2: indent 2 is not a multiple of 4"]
assert check_indentation('def f():\n    return 1\n') == []

---

## Solution 7 — Parse an indented log into a tree

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** stacks, indentation, dicts

**Explanation and reasoning**

The pop loop closes every open level that is not shallower than the new line, which is exactly the dedent rule. Using setdefault means a parent gains a children key only once it actually has a child, so leaves stay clean.

In [ ]:
def parse_blocks(text: str) -> list[dict]:
    roots: list[dict] = []
    stack: list[tuple[int, dict]] = []
    for raw in text.splitlines():
        if not raw.strip() or raw.lstrip().startswith('#'):
            continue
        width = len(raw) - len(raw.lstrip(' '))
        node: dict = {'text': raw.strip()}
        while stack and stack[-1][0] >= width:
            stack.pop()
        if stack:
            stack[-1][1].setdefault('children', []).append(node)
        else:
            roots.append(node)
        stack.append((width, node))
    return roots

**Complexity**

Time O(n) amortised over all lines; space O(d) for the maximum depth.

**Edge cases and failure modes**

Ragged input cannot crash the loop because every pop is guarded.

**Alternative approaches**

A recursive descent version is shorter but risks RecursionError on deep input.

**Tests — run the cell to verify the reference solution**

In [ ]:
out = parse_blocks('a\n  b\n  c\nd')
assert out == [{'text': 'a', 'children': [{'text': 'b'}, {'text': 'c'}]}, {'text': 'd'}]
assert parse_blocks('') == []

---

## Solution 8 — Extract docstrings without importing

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** ast, docstrings, dicts

**Explanation and reasoning**

Parsing rather than importing means the source is inspected as data and never executes. ast.get_docstring handles the indentation cleanup that a raw __doc__ would not, and filtering on the node type restricts the result to functions as required.

In [ ]:
import ast


def extract_docstrings(source: str) -> dict:
    tree = ast.parse(source)
    found: dict = {}
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            found[node.name] = ast.get_docstring(node)
    return found

**Complexity**

Time O(n) in the source length; space O(f) for f functions.

**Edge cases and failure modes**

A function without a docstring maps to None; invalid source raises SyntaxError.

**Alternative approaches**

inspect would require importing the module and running its code.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert extract_docstrings('def f():\n    """Doc."""\n') == {'f': 'Doc.'}
assert extract_docstrings('x = 1\n') == {}

---

## Solution 9 — Split semicolon-packed statements safely

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** parsing, strings, tokenize

**Explanation and reasoning**

The same quote-tracking scan from exercise 1 is reused, so a semicolon inside a string never splits a statement. Empty fragments are dropped before the compound check, which means a trailing semicolon is harmless while a genuine if or for is still rejected.

In [ ]:
COMPOUND = ('if', 'for', 'while', 'def', 'class', 'with', 'try', 'match')


def split_statements(line: str) -> list[str]:
    quote = None
    previous = ''
    parts, current = [], []
    for ch in line:
        if quote:
            current.append(ch)
            if ch == quote and previous != '\\':
                quote = None
        elif ch in ('"', "'"):
            quote = ch
            current.append(ch)
        elif ch == ';':
            parts.append(''.join(current))
            current = []
        else:
            current.append(ch)
        previous = ch
    parts.append(''.join(current))
    statements = [s.strip() for s in parts if s.strip()]
    if statements and statements[0].startswith(COMPOUND):
        raise ValueError('compound statement cannot be split safely')
    return statements

**Complexity**

Time O(n) in the line length; space O(n).

**Edge cases and failure modes**

A trailing semicolon produces no empty statement; leading whitespace does not mask the keyword check.

**Alternative approaches**

tokenize is exact but needs a complete source document.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert split_statements('a = 1; b = 2') == ['a = 1', 'b = 2']
assert split_statements("s = ';' ; t = 2") == ["s = ';'", 't = 2']

---

## Solution 10 — Produce a module outline

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** ast, tuples, reporting

**Explanation and reasoning**

Mapping node classes to labels avoids a chain of isinstance checks and keeps the filter table visible. The summary is taken from the first non-empty docstring line, which gives a usable outline without dumping whole paragraphs into the report.

In [ ]:
import ast


def outline(source: str) -> list[tuple]:
    tree = ast.parse(source)
    rows: list[tuple] = []
    kinds = {ast.FunctionDef: 'def', ast.AsyncFunctionDef: 'def', ast.ClassDef: 'class'}
    for node in tree.body:
        kind = kinds.get(type(node))
        if kind is None:
            continue
        doc = ast.get_docstring(node) or ''
        summary = doc.strip().splitlines()[0] if doc.strip() else ''
        rows.append((kind, node.name, node.lineno, summary))
    return rows

**Complexity**

Time O(n) in the source length; space O(d) for d top-level definitions.

**Edge cases and failure modes**

A definition without a docstring yields an empty summary rather than None.

**Alternative approaches**

tokenize plus a line scan is simpler but cannot tell a class from a function.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert outline('def f():\n    """One.\n    More."""\n') == [('def', 'f', 1, 'One.')]
assert outline('x = 1\n') == []

---

# Robotics challenge solution — ROBO-X Challenge: Bring-Up Log Reader

**Explanation**

Rendering structured text and parsing it straight back proves the parser works on the exact shape the system emits. Splitting the first block's text recovers the robot name while keeping the child lines as raw field strings for the caller to interpret.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'shared').is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402


def parse_blocks(text):
    """Parse space-indented lines into nested dicts (self-contained)."""
    roots, stack = [], []
    for raw in text.splitlines():
        if not raw.strip() or raw.lstrip().startswith('#'):
            continue
        width = len(raw) - len(raw.lstrip(' '))
        node = {'text': raw.strip()}
        while stack and stack[-1][0] >= width:
            stack.pop()
        (stack[-1][1].setdefault('children', []).append(node)
         if stack else roots.append(node))
        stack.append((width, node))
    return roots


def render_status(robot) -> str:
    state = robot.status()
    return (f"robot {state['name']}\n"
            f"    battery {state['battery_pct']:.1f}%\n"
            f"    distance {state['travelled_m']:.2f} m")


def summarise(robot) -> dict:
    """Parse our own indented status log back into data."""
    blocks = parse_blocks(render_status(robot))
    return {
        'name': blocks[0]['text'].split(' ', 1)[1],
        'fields': [c['text'] for c in blocks[0].get('children', [])],
    }

**Complexity**

Time O(n) in the log length; space O(n).

**Notes and trade-offs**

A robot with no movement reports a zero distance line that still parses.

In [ ]:
robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
out = summarise(robot)
assert out['name'] == 'rover-01'
assert len(out['fields']) == 2

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.